# ECMWF Open Data Access Example

This notebook demonstrates how to access and visualize ECMWF IFS forecast data from the zarr store.

Inspired by [dynamical.org](https://dynamical.org) data access patterns.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional: for nicer maps
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    print("Cartopy not available. Install with: pip install cartopy")

## 1. Open the Dataset

Open the zarr store. Use the GitHub Pages URL for remote access, or local path for local testing.

In [ ]:
# For local access
zarr_path = "../data/ecmwf/ifs/forecast-15-day/latest.zarr"

# For remote access (after deploying to GitHub Pages)
# zarr_path = "https://YOUR_USERNAME.github.io/ecmwf_open_data_to_zarr/data/ecmwf/ifs/forecast-15-day/latest.zarr"

ds = xr.open_zarr(zarr_path, decode_timedelta=True, chunks=None)
ds

## 2. Single Point Selection

Extract forecast for a specific location (e.g., London, UK)

In [ ]:
# Select latest forecast
latest_init = ds.init_time.max().values
print(f"Latest forecast initialized at: {latest_init}")

# Select London (51.5°N, 0.1°W -> 359.9°E)
london = ds.sel(
    init_time=latest_init,
    latitude=51.5,
    longitude=359.9,  # -0.1° W = 359.9° E
    method="nearest"
)

london

## 3. Temperature Time Series

Plot the temperature forecast for London

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Convert Kelvin to Celsius
temp_c = london["temperature_2m"] - 273.15

temp_c.plot(ax=ax, linewidth=2)
ax.set_title(f"London Temperature Forecast (Init: {pd.Timestamp(latest_init).strftime('%Y-%m-%d %H:%M UTC')})")
ax.set_ylabel("Temperature (°C)")
ax.set_xlabel("Lead Time")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Wind Speed Calculation

Calculate and plot wind speed from U and V components

In [ ]:
# Calculate wind speed
wind_speed = np.sqrt(london["wind_u_10m"]**2 + london["wind_v_10m"]**2)

fig, ax = plt.subplots(figsize=(12, 6))
wind_speed.plot(ax=ax, linewidth=2, color="steelblue")
ax.set_title(f"London 10m Wind Speed Forecast")
ax.set_ylabel("Wind Speed (m/s)")
ax.set_xlabel("Lead Time")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Multiple Variables Dashboard

Create a dashboard showing multiple forecast variables

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

# Temperature
(london["temperature_2m"] - 273.15).plot(ax=axes[0], linewidth=2, color="red")
axes[0].set_title("Temperature (°C)")
axes[0].grid(True, alpha=0.3)

# Precipitation (convert m to mm)
(london["total_precipitation"] * 1000).plot(ax=axes[1], linewidth=2, color="blue")
axes[1].set_title("Total Precipitation (mm)")
axes[1].grid(True, alpha=0.3)

# Wind speed
wind_speed.plot(ax=axes[2], linewidth=2, color="green")
axes[2].set_title("Wind Speed (m/s)")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Regional Selection

Select a region (e.g., Europe) and visualize

In [ ]:
# Select Europe region and 24-hour lead time
europe = ds.sel(
    init_time=latest_init,
    lead_time="24h",
    latitude=slice(70, 35),    # 70°N to 35°N
    longitude=slice(350, 40)    # 10°W to 40°E (350° to 40°)
)

europe

## 7. Geographic Visualization

Create a map of temperature at 24-hour lead time

In [ ]:
if HAS_CARTOPY:
    fig, ax = plt.subplots(
        figsize=(14, 10),
        subplot_kw={"projection": ccrs.PlateCarree()}
    )
    
    # Convert temperature to Celsius
    temp_map = europe["temperature_2m"] - 273.15
    
    # Plot
    im = temp_map.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap="RdYlBu_r",
        cbar_kwargs={"label": "Temperature (°C)", "shrink": 0.8}
    )
    
    # Add features
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.LAND, alpha=0.1)
    ax.gridlines(draw_labels=True, alpha=0.3)
    
    ax.set_title(f"Europe Temperature Forecast (+24h)\nInit: {pd.Timestamp(latest_init).strftime('%Y-%m-%d %H:%M UTC')}")
    plt.tight_layout()
    plt.show()
else:
    # Simple plot without cartopy
    fig, ax = plt.subplots(figsize=(12, 8))
    (europe["temperature_2m"] - 273.15).plot(
        ax=ax,
        cmap="RdYlBu_r",
        cbar_kwargs={"label": "Temperature (°C)"}
    )
    ax.set_title(f"Europe Temperature Forecast (+24h)")
    plt.tight_layout()
    plt.show()

## 8. Multi-Location Comparison

Compare forecasts for multiple cities

In [ ]:
cities = {
    "London": (51.5, 359.9),
    "Paris": (48.9, 2.4),
    "Berlin": (52.5, 13.4),
    "Madrid": (40.4, 356.3),  # -3.7° W = 356.3° E
    "Rome": (41.9, 12.5),
}

fig, ax = plt.subplots(figsize=(14, 8))

for city_name, (lat, lon) in cities.items():
    city_data = ds.sel(
        init_time=latest_init,
        latitude=lat,
        longitude=lon,
        method="nearest"
    )
    
    temp_c = city_data["temperature_2m"] - 273.15
    temp_c.plot(ax=ax, label=city_name, linewidth=2)

ax.set_title(f"Temperature Forecast Comparison (Init: {pd.Timestamp(latest_init).strftime('%Y-%m-%d %H:%M UTC')})")
ax.set_ylabel("Temperature (°C)")
ax.set_xlabel("Lead Time")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Data Statistics

Compute statistics across the forecast

In [ ]:
# Global statistics for latest forecast
latest_forecast = ds.sel(init_time=latest_init, lead_time="24h")

print("Global Statistics (24-hour forecast):")
print("=" * 50)

for var in ["temperature_2m", "wind_u_10m", "wind_v_10m", "surface_pressure"]:
    if var in latest_forecast:
        data = latest_forecast[var]
        print(f"\n{var}:")
        print(f"  Min:  {float(data.min().values):.2f}")
        print(f"  Max:  {float(data.max().values):.2f}")
        print(f"  Mean: {float(data.mean().values):.2f}")
        print(f"  Std:  {float(data.std().values):.2f}")

## 10. Advanced: Hovmöller Diagram

Show how temperature evolves along a latitude line over time

In [ ]:
# Select data along 50°N (mid-latitudes)
lat_slice = ds.sel(
    init_time=latest_init,
    latitude=50,
    method="nearest"
)

# Plot temperature as function of longitude and lead time
fig, ax = plt.subplots(figsize=(14, 8))

temp_hovmoller = lat_slice["temperature_2m"] - 273.15
temp_hovmoller.plot(
    x="longitude",
    y="lead_time",
    ax=ax,
    cmap="RdYlBu_r",
    cbar_kwargs={"label": "Temperature (°C)"}
)

ax.set_title(f"Temperature Hovmöller Diagram (50°N)\nInit: {pd.Timestamp(latest_init).strftime('%Y-%m-%d %H:%M UTC')}")
ax.set_ylabel("Lead Time")
ax.set_xlabel("Longitude (°E)")
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
- Opening zarr datasets with xarray
- Point selection using `sel()` with `method="nearest"`
- Regional subsetting using `slice()`
- Time series visualization
- Geographic mapping (with cartopy)
- Multi-location comparisons
- Statistical analysis

Following the patterns from [dynamical.org](https://dynamical.org) examples.